# Neural Network Training

- [Activation Functions](#Activation-Functions)
  - [Why non-linear activations are needed at all](#Why-non-linear-activations-are-needed-at-all)
  - [Sigmoid and Tanh, worked at the same input to show saturation](#Sigmoid-and-Tanh,-worked-at-the-same-input-to-show-saturation)
  - [ReLU, and why it dominates deep networks](#ReLU,-and-why-it-dominates-deep-networks)
  - [The dead ReLU problem, and Leaky ReLU's fix](#The-dead-ReLU-problem,-and-Leaky-ReLU's-fix)
  - [GELU, briefly](#GELU,-briefly)
  - [Softmax, cross-referenced](#Softmax,-cross-referenced)
- [PyTorch Basics](#PyTorch-Basics)
  - [Setup and devices](#Setup-and-devices)
  - [Tensors](#Tensors)
  - [Common tensor operations](#Common-tensor-operations)
  - [Tensor manipulation and shapes](#Tensor-manipulation-and-shapes)
  - [Computational graph and autograd](#Computational-graph-and-autograd)
  - [Autograd, verified against a hand-derived gradient](#Autograd,-verified-against-a-hand-derived-gradient)
  - [Backpropagation: the chain rule, layer by layer](#Backpropagation:-the-chain-rule,-layer-by-layer)
  - [Model parameters](#Model-parameters)
- [Building and Training Networks](#Building-and-Training-Networks)
  - [nn.Module](#nn.Module)
  - [A multilayer network](#A-multilayer-network)
  - [Modules and containers](#Modules-and-containers)
  - [Data loaders](#Data-loaders)
  - [The training loop](#The-training-loop)
  - [A simple regression network](#A-simple-regression-network)
  - [Example: a self-attention layer](#Example:-a-self-attention-layer)
  - [Saving and loading models](#Saving-and-loading-models)
- [Training Mechanics](#Training-Mechanics)
  - [Weight initialization](#Weight-initialization)
  - [Optimizers, worked by hand](#Optimizers,-worked-by-hand)
  - [Learning rate schedules](#Learning-rate-schedules)
  - [Dropout](#Dropout)
  - [Batch normalization](#Batch-normalization)
  - [Residual connections](#Residual-connections)
  - [Exploding gradients and gradient clipping](#Exploding-gradients-and-gradient-clipping)
  - [Mini-batches and batch size](#Mini-batches-and-batch-size)
  - [Early stopping and data augmentation](#Early-stopping-and-data-augmentation)
- [Distributed Training](#Distributed-Training)
  - [Data parallelism (DDP)](#Data-parallelism-(DDP))
  - [Model parallelism vs. pipeline parallelism](#Model-parallelism-vs.-pipeline-parallelism)
  - [Sync vs. async gradient aggregation](#Sync-vs.-async-gradient-aggregation)
- [Transfer, Compression and Privacy](#Transfer,-Compression-and-Privacy)
  - [Transfer learning](#Transfer-learning)
  - [Model compression: pruning, quantization and distillation](#Model-compression:-pruning,-quantization-and-distillation)
  - [Federated learning](#Federated-learning)

### Activation Functions

Sigmoid and softmax already have detailed treatment elsewhere (`08-classical-ml.ipynb`'s Logistic Regression section, and its numerically-stable-softmax appendix), referenced not repeated. This notebook is the comparison across the whole family, worked through the same input value so the vanishing-gradient contrast is directly visible.

#### Why non-linear activations are needed at all

Stacking linear layers without a non-linearity between them collapses to a single linear layer, no matter how many layers deep: if layer 1 computes W1*x and layer 2 computes W2*(W1*x), that is just (W2*W1)*x, a single matrix multiplication, all the extra layers bought zero additional representational power. A non-linear activation between layers is what lets a deep network represent genuinely non-linear functions, without it, "deep" learning would be mathematically identical to a single linear model regardless of depth.

#### Sigmoid and Tanh, worked at the same input to show saturation

Sigmoid: sigma(z) = 1/(1+e^-z), range (0,1), derivative = sigma(z)*(1-sigma(z)).
Tanh: range (-1,1), zero-centered (an advantage over sigmoid, whose outputs are always positive, which can bias gradient directions in later layers), derivative = 1 - tanh(z)^2.

Worked at z=3 (a moderately large positive input, not even extreme):
```
sigmoid(3) = 0.9526,  sigmoid'(3) = 0.9526*(1-0.9526) = 0.0452
tanh(3)    = 0.9951,  tanh'(3)    = 1 - 0.9951^2       = 0.0098
```
Both gradients are already small at z=3, and both continue shrinking toward 0 as z grows further, this is saturation, the exact mechanism behind the vanishing gradient problem covered in `04-nlp.ipynb`'s RNN section, repeated multiplication by numbers this small across many layers or timesteps drives the gradient toward zero. Tanh saturates even faster than sigmoid here (0.0098 vs 0.0452), despite being zero-centered.

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_grad(z):
    s = sigmoid(z)
    return s * (1 - s)

def tanh_grad(z):
    return 1 - np.tanh(z) ** 2

z = 3.0
print(f"sigmoid({z}) = {sigmoid(z):.4f}, gradient = {sigmoid_grad(z):.4f}")
print(f"tanh({z}) = {np.tanh(z):.4f}, gradient = {tanh_grad(z):.4f}")

#### ReLU, and why it dominates deep networks

Formula: relu(z) = max(0, z). Derivative: 1 for z>0, 0 for z<0 (undefined exactly at 0, conventionally set to 0 or 1 by implementation).

Worked at the SAME z=3 used above:
```
relu(3) = 3
relu'(3) = 1     <- full gradient, no saturation at all for positive inputs
```
Compare to sigmoid's 0.0452 and tanh's 0.0098 at the identical input, ReLU's gradient does not shrink at all as z grows, for any positive input, the gradient is always exactly 1. This is the core reason ReLU became the default for deep networks, no saturation on the positive side means gradients can flow through many layers without vanishing, directly solving the problem sigmoid/tanh created.

In [ ]:
def relu(z):
    return np.maximum(0, z)

def relu_grad(z):
    return (z > 0).astype(float)

print(f"relu({z}) = {relu(z)}, gradient = {relu_grad(np.array(z))}")

#### The dead ReLU problem, and Leaky ReLU's fix

ReLU's weakness is symmetric to its strength: for NEGATIVE inputs, both the output and the gradient are exactly 0.
```
relu(-2) = 0
relu'(-2) = 0    <- exactly zero, no gradient signal at all
```
If training pushes a neuron's weights such that its input is consistently negative across the whole dataset, that neuron outputs 0 for everything and receives a gradient of exactly 0 forever after, it is "dead," permanently inactive, with no mechanism to ever recover (a zero gradient means no weight update can happen through that path).

Leaky ReLU fixes this with a small non-zero slope for negative inputs instead of a hard zero: leaky_relu(z) = z if z>0, else alpha*z (alpha commonly 0.01).
```
leaky_relu(-2) = 0.01 * -2 = -0.02
leaky_relu'(-2) = 0.01           <- small, but NOT zero, the neuron can still receive a gradient signal and recover
```
ELU (Exponential Linear Unit) is a smoother variant of the same fix, using an exponential curve instead of a straight line for negative inputs, smoother gradients near the transition point at the cost of being more expensive to compute (involves exp()).

In [ ]:
def leaky_relu(z, alpha=0.01):
    return np.where(z > 0, z, alpha * z)

def leaky_relu_grad(z, alpha=0.01):
    return np.where(z > 0, 1.0, alpha)

z_neg = -2.0
print(f"relu({z_neg}) = {relu(z_neg)}, gradient = {relu_grad(np.array(z_neg))}")
print(f"leaky_relu({z_neg}) = {leaky_relu(z_neg):.3f}, gradient = {leaky_relu_grad(z_neg)}")

#### GELU, briefly

Smooth approximation combining ReLU's shape with a probabilistic weighting (multiplies the input by the standard Normal CDF evaluated at that input, connecting back to `02-stats.ipynb`'s Normal distribution section), used as the default activation in most modern transformer architectures (see `05-tokens-arch.ipynb`), the smoothness (no hard kink at 0, unlike ReLU) tends to help optimization in very deep networks, at extra compute cost per activation compared to plain ReLU.

#### Softmax, cross-referenced

Full mathematical treatment, the numerically-stable implementation, and worked multi-class examples already covered in `08-classical-ml.ipynb` (Logistic Regression section) and reused throughout `09-trees.ipynb`'s multi-class XGBoost walkthrough. Not repeated here, the short version: softmax is the multi-class generalization of sigmoid, turning K raw scores into K probabilities that sum to 1, used as the FINAL layer activation for multi-class classification, never as a hidden-layer activation the way the functions above are.

In [ ]:
import torch
import torch.nn as nn

z_tensor = torch.tensor([3.0, -2.0, 0.5])

for name, fn in [("sigmoid", nn.Sigmoid()), ("tanh", nn.Tanh()), ("ReLU", nn.ReLU()),
                 ("LeakyReLU", nn.LeakyReLU(0.01)), ("ELU", nn.ELU()), ("GELU", nn.GELU())]:
    print(f"{name}({z_tensor.tolist()}) = {fn(z_tensor).tolist()}")

### PyTorch Basics

#### Setup and devices

In [ ]:
import torch

print(torch.__version__) # PyTorch Version
print(torch.mps.is_available()) # For Mac OS - True
print(torch.backends.mps.is_available()) # For Mac OS - True
print(torch.mps.device_count()) # No of devices, - Only 1
print("mps" if torch.mps.is_available() else "cpu")

# ========== MPS / CUDA & GPU ================
# 1. Only 1 device, MPS backend treats entire GPU as single device
# 2. No Multi-GPU Support, CUDA's alternative is MPS
# 3. True Multiple GPUS is possible on system with discrete GPUs (NVIDIA cards)
# 4. To check cores - About this Mac > More info > Hardware ?
# 4.1 Run on terminal `system_profiler SPDisplayDataType`

# Mac has an Apple Silicon chip, to accelerate PyTorch code.

2.7.0
True
True
1
mps


In [22]:
if torch.mps.is_available():
    device = torch.device("mps")
    x = torch.randn(3, 3).to(device)
    print("Running on MPS:", x.device)
else:
    print("MPS not available")

if torch.cuda.is_available():
    device = torch.device("cuda")
    x = torch.randn(3, 3).to(device)
    print("Running on CUDA:", x.device)
else:
    print("CUDA not available")

Running on MPS: mps:0
CUDA not available


#### Tensors

Like numpy arrays, with two things numpy does not have: GPU support (`.to("cuda")` moves computation onto a GPU with no code change beyond that), and autograd (tensors can track the operations applied to them, enabling automatic gradient computation, covered next).

#### Common tensor operations

In [ ]:
# 1. Tensors are data-containers for array-like
tensor0d = torch.tensor(1) # 0d Tensor
tensor1d = torch.tensor([1, 2, 3]) # 1D Tensor
tensor2d = torch.tensor([[1, 2, 3], [3, 4, 4]]) # 2D Tensor, From Nested Python List
tensor3d = torch.tensor([[[1, 2], [3, 4]], [[1, 6], [2, 9]]])

# 2. Tensor DataTypes
print(tensor1d.dtype)

# Float Data Type
floatvec = torch.tensor([1.0, 3.0, 5.0, 6.9])
print(floatvec.dtype) # 32-bit 
# ===============
# A 32 bit offers sufficient precision, consume less memory & resources
# Most GPU Arch are optimized for 32-Bits computations.
# Hence Speed Up Model Trainig & inference
# ===============

# Possible to change the precision using `.to` method
floatvec = tensor1d.to(torch.float32)
print(floatvec.dtype)

# 3 Tensor Operations
print(tensor2d)
print(tensor2d.shape) # Tensor has 2 rows and 3 columns
print(tensor2d.reshape(3, 2))

print(tensor3d.reshape(4, -1).shape) 
# Use of -1 to let PyTorch infer correct dimension

# Check `.is_contiguous()` for checking memory allocation
# Use .contiguous() to get a contiguous copy if needed 
# (e.g., before using .view() on a non-contiguous tensor).
tensor2d.view(3, 2) # Most common way to reshape
tensor2d.T # Transpose the tensor, flipping it across its diagonal
print(tensor2d.transpose(0, 1)) # Swaps dim 0 and dim 1
print(tensor2d.T.is_contiguous()) # False - Changes in logical view

# Matrix Multiplication
tensor2d.matmul(tensor2d.T) # matmul or @
tensor2d @ tensor2d.T

torch.int64
torch.float32
torch.float32
tensor([[1, 2, 3],
        [3, 4, 4]])
torch.Size([2, 3])
tensor([[1, 2],
        [3, 3],
        [4, 4]])
torch.Size([4, 2])
tensor([[1, 3],
        [2, 4],
        [3, 4]])
False


tensor([[14, 23],
        [23, 41]])

#### Tensor manipulation and shapes

1. Batching - Models process multiple samples at once (batch dimension)
2. Layer Requirement - Expect inputs in certain shapes
3. Multi-head Attention - Require Splitting & merging dimensions for heads.
4. Broadcasting - Operations like addition/multiplication may require matching shapes

In [ ]:
x = torch.randn(2, 3, 4)  # create a random tensor with shape (2, 3, 4, 5)
# shape (2, 3, 4, 5) means:
# 2 matrices, each with 3 rows and 4 columns, and 
# each element is a vector of size 5
print(x.shape)  # print the tensor
y = x.transpose(0, 1) # transpose the first two dimensions
# Switching betweeen the [batch, seuence, feature] and [sequence, batch, feature] formats
print(y.shape)  # print the transposed tensor

# Again back to [batch, sequence, feature]
y = y.permute(1, 0, 2)
print(y.shape)

torch.Size([2, 3, 4])
torch.Size([3, 2, 4])
torch.Size([2, 3, 4])


In [ ]:
# Reshaping / view
# Changes shape of tensor without changing its data.
# Used to flatten images, prepare batches,

x = torch.arange(6) # Shape [6]
x_reshaped = x.view(2, 3) # Shape [2, 3]

# Use -1 to let library infer correct dimension. x.view(-1, 3)

# 1. Flattening for Fully Connected Layers
# linear layer expects [batch, features], not [batch, channesl, height, width]
x = torch.randn(32, 3, 28, 28) # [batch, channesl, height, width]
x_flat = x.view(32, -1) # [batch, features]

# 2. Adding a Batch Dimension
# if you have single sample but model expect a batch
x = torch.randn(10) # [features]
x_batch = x.unsqueeze(0) # [1, features]

# 3. Preparing Sequnces for RNNs
# PyTorch RNNs expect [seq, batch, features]
x = torch.randn(64, 10, 128) # [batch, seq, features]
x_seq_first = x.permute(1, 0, 2) # [seq, batch, features]

# 4. unsqueeze - adds a new dim of size 1 at specified position (axis)
# enables broadcasting, expected input shape of a layer
x = torch.tensor([1, 2, 3]) # Shape [3]
x1 = x.unsqueeze(0) # shape: [1, 3]
x2 = x.unsqueeze(1) # shape: [3, 1]

# Suppose model expects [batch, features], but you have a single feature

# 5. squeeze - removes all dim of size
# reduce rank of tensor oprations
y = torch.rann(1, 3, 1, 5)
y1 = y.squeeze() # Shape: [3, 5], (removes all size-1 dimes)
y2 = y.squeeze(2) # Shape: [1, 3, 5] (removes only dim 2)

In [ ]:
# 6. cat - joins a squence of tensor along an existing dims
a = torch.randn(2, 3)
b = torch.randn(2, 3)

cat0 = torch.cat([a, b], dim=0) # concatenate along rows (dim=0): shape [4, 3]
cate1 = torch.cat([a, b], dim=1) # along columns (dim=1): shape [2, 6]

# 7. stack - Squence of tensor along a new dim
stack0 = torch.stack([a, b], dim=0) # row, result shape [2, 2, 3]
stack1 = torch.stack([a, b], dim=1) # column, shape[2, 2, 3]

In [ ]:
# 8. split - divide a tensor into a list of smaller tensor of specified size(s)
# useful for dividing data into mini-bath, splitting features
x = torch.arnage(12).reshape(3, 4) # Shape [3, 4]
splits = torch.split(x, 2, dim=1) # column, split into 2. # [3, 2] [3, 2]

# 9. chunnk - divide into specified number of equal chunks
chunks = torch.chunk(x, 2, dim=0) # 2 parts along rows. #[2, 4] [1, 4]

In [ ]:
# 10. Reduction Operations (sum, mean, max, min)
# collapse one or more dims. Specify the dim to reduce
x = torch.tensor([[1., 2.], [3., 4.]])
total = x.sum()
row_sum = x.sum(dim=0) # [4., 6.]
col_mean = x.mean(dim=1) #[1.5, 3.5]
max_val, max_idx = x.max(dim=1) #([2., 4.], [1, 1]) # max val & its index

# loss calculation, loss = (pred - target).pow(2).mean()
# pooling layers: torch.maxpool2d
# Normalization: x - x.mean(dim=0)

In [14]:
# 11. Matrix Multiplications, Dot Prodction
# Use @ or torch.matmul for matrix multiplication
# Use torch.dot for 1D vectors
# Linear Layer = x @ W.T + b
# Attention Score = Q @ K.T

a = torch.randn(2, 3)
b = torch.randn(3, 4)

# Matrix Multiplication: [2, 3] @ [3, 4] -> [2, 4]
c = a @ b

# Dot Prodcut : [3], [3] -> Scalar
v1= torch.tensor([1., 2., 3.])
v2 = torch.tensor([4., 5., 6.])
dot = torch.dot(v1, v2)


#### Computational graph and autograd

In [37]:
# PyTorch’s automatic differentiation engine, also known as autograd
# to compute gradients in dynamic computational graphs automatically

# computation graph  -> lays out the sequence of calculations needed to 
# compute the output of a neural network – would be required 
# to compute the required gradients for backpropagation, 
# which is the main training algorithm for neural networks.
import torch.nn.functional as F

y = torch.tensor([1.0]) # True label
x1 = torch.tensor([1.1]) # Input Feature
w1 = torch.tensor([2.2]) # weight parameter
b1 = torch.tensor([0.0]) # bias unit

z = x1 * w1 + b1 # net input
a = torch.sigmoid(z) # sigmoid activation

loss = F.binary_cross_entropy(a, y) # predicted probab vs true label
# BCE(a, y) = - [y.log(a) + (1-y).log(1-a)]
# P(y|a) = a^y.(1-a)^(1-y), Bernoulli's PMF
print(loss)

# Can use gradient of loss function w.r.t w1 & b1 (model parameters), 
# to train model


tensor(0.0852)


In [ ]:
# What is sigmoid activation ?
# - 1 / ( 1 + e^(-x)). Real-value to range of 0 to 1. 
# large positive approaches 1, large negative approaches 0. 

# Is it symmetry ? 
# The sigmoid function is not symmetric about the y-axis (not an even function),
# but it is symmetric about the point (0, 0.5)

x = torch.tensor([-50.0])
y = torch.sigmoid(x) # Output - 0
print(y)

tensor([1.9287e-22])


In [23]:
# build such a graph internally by default if one of its terminal nodes 
# has the requires_grad attribute set to True.

# Gradients are required when training neural networks
# via the popular backpropagation algorithm

# Partial Derivative - Rate at which function changes w.r.t to one of its variables
# Gradient - Vector of Partial Derivative of mutivariate function
# Provides info to update each of the parameter that minimizes loss function (gradient descent)
# loss function serves as proxy for the model performance, 

# PyTorch’s autograd engine constructs a computational graph 
# in the background. Then, calling the grad function, 
# we can compute the gradient of the loss with respect to model parameter w1 

import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b1 = torch.tensor([0.0], requires_grad=True)

z = w1 * x1 + b1
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a, y)
grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b1 = grad(loss, b1, retain_graph=True)

# PyTorch destroys the computation graph after calculating the gradients 
# to free memory, hence `retain_graph=True`

print(grad_L_w1)
print(grad_L_b1)

# we can call .backward on the loss, and PyTorch will compute 
# the gradients of all the leaf nodes in the graph, 
# which will be stored via the tensors’ .grad attributes:

loss.backward()
print(w1.grad)
print(b1.grad)

(tensor([-0.0898]),)
(tensor([-0.0817]),)
tensor([-0.0898])
tensor([-0.0817])


#### Autograd, verified against a hand-derived gradient

This is the actual mechanism behind every gradient formula worked out by hand across this series (logreg's `(p-y)*x` in `08-classical-ml.ipynb`, XGBoost's `g=p-y` in `09-trees.ipynb`), PyTorch computes them automatically via the chain rule applied to whatever operations were run on tensors with `requires_grad=True`.

Worked check: same single-point logistic regression setup as `08-classical-ml.ipynb`'s from-scratch gradient derivation. x=2.0, w=0.5, b=0.1, true label y=1.
```
z = w*x + b = 0.5*2 + 0.1 = 1.1
p = sigmoid(1.1) = 0.7503
loss (BCE) = -log(p) = 0.2872   (y=1, so only the log(p) term survives)

hand-derived gradient: dL/dw = (p-y)*x = (0.7503-1)*2 = -0.4994
                        dL/db = (p-y)*1 = -0.2497
```
If autograd is doing the same chain-rule computation the derivation did by hand, `w.grad` and `b.grad` after `.backward()` should come out to those exact numbers.

In [ ]:
import torch

x = torch.tensor(2.0)
y = torch.tensor(1.0)
w = torch.tensor(0.5, requires_grad=True)
b = torch.tensor(0.1, requires_grad=True)

z = w * x + b
p = torch.sigmoid(z)
loss = -(y * torch.log(p) + (1 - y) * torch.log(1 - p))

loss.backward()

print("p:", p.item(), "| loss:", loss.item())
print("autograd dL/dw:", w.grad.item(), "(hand-derived: -0.4994)")
print("autograd dL/db:", b.grad.item(), "(hand-derived: -0.2497)")

#### Backpropagation: the chain rule, layer by layer

A deep network is a long chain of nested functions, loss depends on the last layer's output, which depends on the second-to-last layer's output, and so on back to the input. Backpropagation computes the gradient with respect to EVERY layer's weights by applying the chain rule repeatedly, working backward from the loss, reusing each layer's local gradient as it goes, exactly the same 3-step chain multiplication worked out by hand above, just repeated once per layer instead of once. This is precisely what `.backward()` automates in `01-nn-training.ipynb`, it never derives a closed-form formula the way the hand-worked examples in this series did, it mechanically applies the chain rule at every operation that was run, in reverse order.

#### Model parameters

In [10]:
import torch.nn as nn

class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(4, 2)

# PyTorch makes two tensors for the weights and biases.
# Special because Pytorch marks them as things it should changes during Training.
# When we call model.parameters(), it returns these tensors.
model = MyModel()
for param in model.parameters():
    print(param.shape)
    print(param)

torch.Size([2, 4])
Parameter containing:
tensor([[-0.2013,  0.1039,  0.1993,  0.4580],
        [ 0.1079, -0.2671,  0.1110, -0.2548]], requires_grad=True)
torch.Size([2])
Parameter containing:
tensor([-0.0651, -0.4072], requires_grad=True)


In [ ]:
import torch
import torch.nn as nn

w = nn.Parameter(torch.randn(2, 2))
print(isinstance(w, nn.Parameter))

# nn.Parameter is a special kind of tensor that is automatically registered as a parameter in the module.
# It is used to define learnable parameters in a neural network.
# nn.Parameter is a subclass of torch.Tensor, so it behaves like a tensor.

# If you add this to a module, it will show up in .parameters()
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.my_weight = nn.Parameter(torch.randn(2, 2))

model = MyModel()
print(list(model.parameters()))


### Building and Training Networks

#### nn.Module

Wrapping weights and the forward computation into a class, sklearn's/this series' `LogisticRegression()`, `RandomForestClassifier()` etc. all follow a similar fit/predict pattern, `nn.Module` is PyTorch's version, you define `forward()` (what to compute) and the framework handles `backward()` (how to differentiate it) automatically via autograd.

In [ ]:
import torch.nn as nn

class LogisticRegressionModel(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.linear = nn.Linear(n_features, 1)  # holds w and b internally

    def forward(self, x):
        return torch.sigmoid(self.linear(x))

model = LogisticRegressionModel(n_features=2)
sample_input = torch.tensor([[1.0, 2.0]])
print("model output (untrained, random weights):", model(sample_input))
print("parameters:", list(model.parameters()))

#### A multilayer network

In [ ]:
import torch
class NeuralNetwork(torch.nn.Module):
    # torch.nn.Module has a __call__ method. 
    # gets invoked when nn.Module instance - model(X)
    # __call__ method is responsible for calling `forward` method
    # 1. Hooks - pre-forward & post-forward hooks, fn to register executed before and after 
    # 2. Parameter Checks, # 3. Automatic Differntiation Setup # 4. Device Mgmt, if already inp & output moved

    def __init__(self, num_in, num_out):
        super().__init__()

        self.layers = torch.nn.Sequential(

            torch.nn.Linear(num_in, 30),
            torch.nn.ReLU(),

            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            torch.nn.Linear(20, num_out),
            # Output of last layer
            # No passing to a nonlinear activation fn.
            # combine the softmax operation with negative log-likelihood loss in a single class
            # due to numerical efficiency and stability


        )
    def forward(self, x):
        logits = self.layers(x) # As Sequential is already part of __init__
        return logits

#=========Model Arch ============#
model = NeuralNetwork(50, 3)
print(model)

#============ Parameters =========#
num_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
print("Total number of trainable model parameters : ", num_params)
# each parameter which requires_grad=True, count as trainable parameter
# these are contained in nn.Linear layers (Fully Connected Layer)
print(model.layers[0].weight.shape)
# model weights are initalized with small random numbers - to break symmetry during training

#========== Model Call ============#
torch.manual_seed(123)
X = torch.rand((1, 50)) # a single random training example with 50 features
out = model(X) # it automatically executes the forward pass of the model ?
print(out)

# Returns three scores, and grad_fn - Which is used by PyTorch to compute gradients
# If we just use for prediction after training, constructing CP for backpropogation can be wasteful
# unnecessary computations and consumes additional memory
# Hence use torch.no_grad() context manager, does not keep track of gradient

#========== Inference =============#
with torch.no_grad():
#    out = model(X)
    out = torch.softmax(model(X), dim=1)
print(out)
# The values can be interpreted as class-membership that sum up to 1

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)
Total number of trainable model parameters :  2213
torch.Size([30, 50])
tensor([[-0.0879,  0.1729,  0.1534]], grad_fn=<AddmmBackward0>)
tensor([[0.2801, 0.3635, 0.3565]])


#### Modules and containers

In [ ]:
import torch
import torch.nn as nn

class MyModule(nn.Module):
    def __init__(self, num_layers, input_dim, output_dim):
        super().__init__()
        # Module holds a list of layers, each is a linear layer
        self.layers = nn.ModuleList(
            [nn.Linear(input_dim, output_dim) for _ in range(num_layers)]
        )
    def forward(self, x):
        # Iterate through each layer in ModuleList
        for layer in self.layers:
            x = layer(x)
        return x

# ModuleList register each layer as a submoudle,
# so their parameters are included in model.parameters() 

In [2]:
model = nn.Sequential(
    nn.Linear(10, 20),
    nn.ReLU(),
    nn.Linear(20, 30)
)

# Sequential - to define a model as a sequence of layers.
# It is a subclass of nn.Module that allows you to stack layers in a sequential manner.
print(model)

Sequential(
  (0): Linear(in_features=10, out_features=20, bias=True)
  (1): ReLU()
  (2): Linear(in_features=20, out_features=30, bias=True)
)


In [3]:
class DictModel(nn.Module):
    def __init__(self):
        super().__init__()
        # ModuleDict holds named layers
        self.layers = nn.ModuleDict(
            {
                "fc1": nn.Linear(10, 20),
                'fc2': nn.Linear(20, 5)
            }
        )
    def forward(self, x):
        x = self.layers['fc1'](x)
        x = self.layers['fc2'](x)
        return x

# ModuleDict is useful when you want to access layers by name
model = DictModel()
print(model)

DictModel(
  (layers): ModuleDict(
    (fc1): Linear(in_features=10, out_features=20, bias=True)
    (fc2): Linear(in_features=20, out_features=5, bias=True)
  )
)


#### Data loaders

In [ ]:
# Dataset class is used to define how each record is loaded
# DataLoader handles how the data is shuffled and assembled into batches
from torch.utils.data import Dataset, DataLoader

X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])
y_train = torch.tensor([0, 0, 0, 1, 1])

X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
])
y_test = torch.tensor([0, 1])

# Class Label Numbering
# - Class labels to start with 0 and largest class label value should not exceed
# - number of output nodes minus 1

# use to instantiate DataLoader
class ToyDataset(Dataset):
    def __init__(self, X, y):
        # setup attributes that can be accessed in __getitem__ or __len__
        # as we created tensor X, y that sits in memoty, simply assigning
        # to our placeholder objects
        self.features = X
        self.labels = y

    def __getitem__(self, index):
        # returning exactly one item from the dataset via index
        one_x = self.features[index]
        one_y = self.labels[index]

        return one_x, one_y

    
    def __len__(self):
        # lenght of dataset
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0, # crucial for parallelizing data laoding & preprocessing
    # as set to 0, data loading done in main process
    # CPU will take time to load & preprocess the data. As result GPU is idle
    # num worker set to greater than - are launched to load data in parallel
    # freeing main process to focus on training model
    
    # On Jupyter Notebook = increasing num_Workers may not provide noticeable speedup
    # potential issues of overhead spinning up multiple workers, hence longer
    # sharing resources between different process, resulting in errors / nb crashes

    # setting num_works=4 leads to optimal performance
    #drop_last=True
)

test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=0
)

for idx, (x, y) in enumerate(train_loader):
    # train_loader iterates over train_ds, each example once = training epoch
    print(f"batch {idx+1}:", x, y)

# since 3rd batch only contains one single example. Smaller batch as last batch
# can disturb the convergence during training. therefore drop_last=True

batch 1: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
batch 2: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
batch 3: tensor([[ 2.7000, -1.5000]]) tensor([1])


#### The training loop

Canonical 5-step loop, the same conceptual loop as every `for round in range(n_estimators)` boosting loop and every from-scratch `for i in range(n_iters)` gradient descent loop in this series, just using autograd instead of a manually derived gradient formula:
```
for epoch in range(n_epochs):
    optimizer.zero_grad()      # clear gradients from the previous step, they accumulate otherwise
    predictions = model(X)     # forward pass
    loss = loss_fn(predictions, y)
    loss.backward()            # autograd computes every gradient
    optimizer.step()           # w -= lr * w.grad, for every parameter, done automatically
```
`optimizer.zero_grad()` matters specifically because PyTorch ACCUMULATES gradients into `.grad` by default (adds to whatever was already there, does not overwrite), useful for some advanced patterns (gradient accumulation across mini-batches too large to fit in memory at once), but means a normal loop that forgets this step silently sums gradients across every epoch, producing wrong, ever-growing updates.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)
X = torch.randn(100, 2)
y = (X[:, 0] + X[:, 1] > 0).float().unsqueeze(1)

model = LogisticRegressionModel(n_features=2)
loss_fn = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

losses = []
for epoch in range(200):
    optimizer.zero_grad()
    preds = model(X)
    loss = loss_fn(preds, y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print("loss at epoch 0:", losses[0])
print("loss at epoch 199:", losses[-1])
print("learned weights:", model.linear.weight.detach())

In [ ]:
import torch.nn.functional as F

torch.manual_seed(123)
model = NeuralNetwork(num_in=2, num_out=2) # 2 input feature , 2 class label
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

# Validation dataset is similar to test set,
# While we only want to use a test set precisely once to avoid
# biasing the evaluation, we use valid set multiple times
num_epochs = 3
for epoch in range(num_epochs):
    model.train() # train mode
    for batch_idx, (features, labels) in enumerate(train_loader):
        logits = model(features)
        # apply softmax function internall
        loss = F.cross_entropy(logits, labels)
        optimizer.zero_grad() # to reset gradiesnt to zero
        # otherwise gradient will accumulate
        loss.backward() # calculate gradients in the CP
        optimizer.step() # Update the model parameters to minimze loss

        # LOGGING
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")
    model.eval()

# since we do not have dropout and regularization
model.eval()
with torch.no_grad():
    outputs = model(X_train)

print(outputs)

torch.set_printoptions(sci_mode=False)
# probas = torch.softmax(outputs, dim=1)
# print(probas)

predictions = torch.argmax(outputs, dim=1)
print(predictions)

torch.sum(predictions == y_train)

def compute_accuracy(model, dataloader):
    model = model.eval()
    correct = 0.0
    total_examples = 0

    for idx, (features, labels) in enumerate(dataloader):

        with torch.no_grad():
            logits = model(features)

        predictions = torch.argmax(logits, dim=1)
        compare = labels == predictions

        correct += torch.sum(compare)
        total_examples += len(compare)

    return (correct / total_examples).item()

compute_accuracy(model, train_loader)
compute_accuracy(model, test_loader)



Epoch: 001/003 | Batch 000/003 | Train/Val Loss: 0.75
Epoch: 001/003 | Batch 001/003 | Train/Val Loss: 0.65
Epoch: 001/003 | Batch 002/003 | Train/Val Loss: 0.42
Epoch: 002/003 | Batch 000/003 | Train/Val Loss: 0.05
Epoch: 002/003 | Batch 001/003 | Train/Val Loss: 0.13
Epoch: 002/003 | Batch 002/003 | Train/Val Loss: 0.00
Epoch: 003/003 | Batch 000/003 | Train/Val Loss: 0.01
Epoch: 003/003 | Batch 001/003 | Train/Val Loss: 0.00
Epoch: 003/003 | Batch 002/003 | Train/Val Loss: 0.02
tensor([[ 2.9320, -4.2563],
        [ 2.6045, -3.8389],
        [ 2.1484, -3.2514],
        [-2.1461,  2.1496],
        [-2.5004,  2.5210]])
tensor([0, 0, 0, 1, 1])


1.0

#### A simple regression network

Same DataLoader + training-loop pattern as above, applied to a regression task instead of
classification: a minimal linear model (y = 2x+1) extended into a real 2-hidden-layer MLP.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(0)

# simple linear model for y = 2x + 1
class SimpleLinear(nn.Module):
    def __init__(self):
        super().__init__()
        self.lin = nn.Linear(1, 1)  # no hidden layer, no ReLU

    def forward(self, x):
        return self.lin(x) # linear Map. X()

model = SimpleLinear()
optimizer = optim.SGD(model.parameters(), lr=1e-2)  # small LR
loss_fn = nn.MSELoss()

# data
x_train = torch.randn(100, 1) * 10.0
y_train = 2.0 * x_train + 1.0 + torch.randn_like(x_train) * 0.5

# training
for epoch in range(200):
    optimizer.zero_grad() # zero accumulated gradient.
    pred = model(x_train)
    loss = loss_fn(pred, y_train)
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0:
        print(f"Epoch {epoch}: Loss={loss.item():.4f}")

# quick check
x_test = torch.tensor([[5.0]])
pred = model(x_test).item()
print(f"Input=5, Pred={pred:.3f}, True={2*5+1}")

Epoch 0: Loss=398.6652
Epoch 20: Loss=85.7614
Epoch 40: Loss=18.6274
Epoch 60: Loss=4.2167
Epoch 80: Loss=1.1203
Epoch 100: Loss=0.4534
Epoch 120: Loss=0.3092
Epoch 140: Loss=0.2777
Epoch 160: Loss=0.2707
Epoch 180: Loss=0.2691
Input=5, Pred=11.026, True=11


In [ ]:
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F

torch.manual_seed(42)
device = ("mps" if torch.mps.is_available() else "cpu")

# MLP with 2 hidden layers and ReLU activation
class MLP(nn.Module):
    def __init__(self, in_dim=1, h1=64, h2=32, out_dim=1):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, h1)
        self.fc2 = nn.Linear(h1, h2)
        self.fc3 = nn.Linear(h2, out_dim)

        # simple initialization (helps stable training)
        nn.init.kaiming_uniform_(self.fc1.weight, nonlinearity='relu')
        nn.init.kaiming_uniform_(self.fc2.weight, nonlinearity='relu')
        nn.init.uniform_(self.fc3.weight, -0.1, 0.1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        return x
    
# generate synthetic linear-ish data with noise (same ground truth: y=2x+1)
N = 2000
x = torch.randn(N, 1) * 10.0
y = 2.0 * x + 1.0 + torch.rand_like(x) * 0.5

# dataset + dataloader for mini-batch SGD
batch_size = 64
dataset = TensorDataset(x, y)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

model = MLP().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# training : mini-batch gradient descent approximates full gradient using batches
epochs = 100
for epoch in range(epochs):
    epoch_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad() # clear gradients from previous batch
        yhat = model(xb) # forward compute predictions
        loss = loss_fn(yhat, yb) # compute batch loss
        loss.backward() # backprop: compute gradienst via autograd
        optimizer.step() # update params 
        epoch_loss += loss.item() * xb.size(0)
    epoch_loss /= N
    if epoch % 10 == 0:
        print(f"Epoch {epoch: 03d}, Loss={epoch_loss:.4f}, Device={device}")

# evaluation on a single example
x_test = torch.tensor([[5.0]]).to(device)
with torch.no_grad():
    pred = model(x_test).cpu().item()
print(f"Input=5, Pred={pred:.3f}, true={2*5+1}")


Epoch  00, Loss=242.8731, Device=mps
Epoch  10, Loss=0.0248, Device=mps
Epoch  20, Loss=0.0233, Device=mps
Epoch  30, Loss=0.0221, Device=mps
Epoch  40, Loss=0.0238, Device=mps
Epoch  50, Loss=0.0234, Device=mps
Epoch  60, Loss=0.0238, Device=mps
Epoch  70, Loss=0.0227, Device=mps
Epoch  80, Loss=0.0228, Device=mps
Epoch  90, Loss=0.0227, Device=mps
Input=5, Pred=11.260, true=11


#### Example: a self-attention layer

In [23]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import functional as F

tokens = ["The", " ", "cat", " ", "sat", " ", "on", " ", "the", " ", "mat", "."]
n_tokens = len(tokens)
d_k = 6

# randomly initialize Q, K, V with Standard Normal distribution (mean=0, std=1)
Q = torch.randn(n_tokens, d_k) # n_tokens x d_k
K = torch.randn(n_tokens, d_k)
V = torch.randn(n_tokens, d_k)

# (n_tokens x d_k) @ (d_k x n_tokens) = (n_tokens x n_tokens)
scores = Q @ K.T 

# Values can become large, so we scale them down by the square root of d_k
# to prevent softmax from saturating
# scaling keeps variance of the dot product more consistent
# (n_tokens x n_tokens) / sqrt(d_k) = (n_tokens x n_tokens)
scaled_score = scores / (d_k ** 0.5)

# softmax to get attention weights last dimension
# For each query, softamx is applied across all keys
# converts each row to probaility distribution
# the last diimension corresponds to the keys
attn_weights = F.softmax(scaled_score, dim=-1)

# (n_tokens x n_tokens) @ (n_tokens x d_k) = (n_tokens x d_k)
# the attention weights are used to weight the values
# the result is a weighted sum of the values
output_original = attn_weights @ V

output_original

tensor([[ 0.3341, -0.5154, -1.2380, -0.2892, -0.4579, -0.2457],
        [-0.6077, -0.0793,  1.2263,  0.4887, -0.1040, -0.6966],
        [-0.2376,  0.6978, -0.2318, -0.5215,  0.0550, -0.2912],
        [-0.1975,  0.7894, -0.4018, -0.5038,  0.0247, -0.5158],
        [ 0.0433,  0.2650, -0.5138, -0.3914, -0.2075, -0.0951],
        [-0.3837,  0.6075,  0.4693, -0.1916, -0.1801, -0.0152],
        [ 0.2334, -0.6505, -1.1035, -0.1337, -0.4387, -0.4416],
        [-0.1326,  0.3576, -0.4958, -0.5872, -0.0889, -0.1419],
        [ 0.3356, -0.6253, -1.3418, -0.2246, -0.4712, -0.3358],
        [-0.1198,  0.5244, -0.3412, -0.5332, -0.0867, -0.0930],
        [-0.0488,  0.1532, -0.6066, -0.5057, -0.1759, -0.2830],
        [-0.1535,  0.2497, -0.4927, -0.4406, -0.1209, -0.2047]])

#### Saving and loading models

In [ ]:
# state_dict is a Python dictionary object that maps
# each layer in the model to its trainable parameters (weights & biases)

# torch.save(model.state_dict(), "model.pth")

# model = NeuralNetwork(2, 2) # needs to match orginaal model exactly
# model.load_state_dict(torch.load("model.pth", weights_only=True))


### Training Mechanics

How a network actually gets trained end to end: tensors, weight initialization, autograd (verified against hand-derived gradients from 01-math.ipynb and 08-classical-ml.ipynb), the training loop, optimizers, and learning rate schedules, worked by hand first, then confirmed with real code. PyTorch is the implementation vehicle used for the code throughout (nn.RNN/nn.LSTM in 04-nlp.ipynb, nn.Dropout in the Dropout section below, nn.MultiheadAttention-style computation in 05-tokens-arch.ipynb all build on the mechanics covered here), not the subject of the notebook itself.

#### Weight initialization

Why not start at zero: if every weight in a layer starts identical (e.g. all zero), every neuron computes the identical output and the identical gradient, they stay identical forever, a symmetry problem, the layer effectively has only 1 unique neuron's worth of learning capacity no matter how many neurons it has.

Why not just large random values: large weights push activations into saturated regions of sigmoid/tanh (see the worked gradient-shrinkage example under Activation Functions), or cause activations to explode layer over layer in a deep network.

Xavier/Glorot initialization (designed for sigmoid/tanh): draw weights so the VARIANCE stays roughly constant from layer to layer, scaled by fan_in and fan_out (the number of inputs and outputs of that layer). Uniform version: limit = sqrt(6/(fan_in+fan_out)), draw uniformly from [-limit, limit].

Worked example, a layer with fan_in=100, fan_out=50:
```
Xavier limit = sqrt(6/(100+50)) = sqrt(0.04) = 0.2
weights drawn uniformly from [-0.2, 0.2]
```
He initialization (designed for ReLU): variance = 2/fan_in, accounts for ReLU zeroing out roughly half of all activations on average (see the dead ReLU section under Activation Functions), so it compensates with a larger variance than Xavier would give, to keep the SURVIVING half of activations at a healthy scale.
```
He variance = 2/100 = 0.02, std = sqrt(0.02) = 0.141
weights drawn from Normal(0, 0.141^2)
```
Rule of thumb: Xavier for sigmoid/tanh layers, He for ReLU layers, PyTorch's `nn.Linear` uses a variant of He-style initialization by default already, this is rarely something you set manually, but knowing why it is not just zeros or arbitrary randomness matters for debugging a network that will not train.

In [ ]:
import torch
import torch.nn as nn

layer = nn.Linear(100, 50)

xavier_layer = nn.Linear(100, 50)
nn.init.xavier_uniform_(xavier_layer.weight)

he_layer = nn.Linear(100, 50)
nn.init.kaiming_normal_(he_layer.weight, nonlinearity="relu")  # "kaiming" is He's other name

print("default init weight std:", layer.weight.std().item())
print("Xavier init weight range: [", xavier_layer.weight.min().item(), ",", xavier_layer.weight.max().item(), "] (expect near +-0.2)")
print("He init weight std:", he_layer.weight.std().item(), "(expect near 0.141)")

In [ ]:
import torch

x = torch.tensor([1.0, 2.0, 3.0])
y = torch.tensor([4.0, 5.0, 6.0])

print("element-wise add:", x + y)
print("dot product:", torch.dot(x, y))
print("shape:", x.shape, "| dtype:", x.dtype)
print("device:", x.device, "| cuda available:", torch.cuda.is_available())

#### Optimizers, worked by hand

SGD: plain gradient descent, `w -= lr * grad`, exactly the update rule derived by hand throughout this series. Reacts to only the CURRENT gradient, no memory of past steps, so it oscillates on noisy gradients.

Momentum: accumulate a running average of past gradients, step using that smoothed direction instead of the raw noisy one. Formula: v = beta*v_prev + grad, then w -= lr*v (beta commonly 0.9).

Worked example, a noisy, oscillating gradient sequence over 3 steps: [1.0, -0.8, 1.2]
```
v1 = 0.9*0 + 1.0 = 1.0
v2 = 0.9*1.0 + (-0.8) = 0.9 - 0.8 = 0.1
v3 = 0.9*0.1 + 1.2 = 0.09 + 1.2 = 1.29
```
Compare the raw gradients [1.0, -0.8, 1.2] (flips sign every step) against the momentum-smoothed v sequence [1.0, 0.1, 1.29] (stays positive throughout, the accumulated positive momentum from step 1 damps step 2's negative gradient rather than fully reversing direction). This is the actual mechanism behind "momentum smooths out noisy updates."

RMSprop: adaptive PER-PARAMETER learning rate, divide the step by a running average of squared gradients, so parameters with consistently large gradients get automatically smaller effective steps (preventing overshoot) and parameters with small gradients get relatively larger steps (preventing them from stalling). Formula: s = beta*s_prev + (1-beta)*grad^2, w -= lr*grad/sqrt(s+eps).

Adam: combines BOTH ideas, momentum (a running average of the gradient itself, called the first moment) and RMSprop (a running average of the squared gradient, the second moment), plus a bias correction for the first few steps (since both running averages start at 0 and are biased low early on). The default choice for training neural networks in practice.

AdamW: Adam plus DECOUPLED weight decay, applies the L2 regularization penalty directly to the weight update, separately from the gradient-based adaptive scaling, rather than folding it into the gradient the way plain Adam does. Fixes a subtle bug-like interaction where Adam's per-parameter adaptive scaling distorts what L2 regularization is supposed to do (shrink every weight by the same proportional amount, see `05-loss-regularization.ipynb`'s Ridge section, Adam's adaptive denominator breaks that uniformity if weight decay is folded into the gradient instead of applied separately).

In [ ]:
gradients = [1.0, -0.8, 1.2]
beta = 0.9
v = 0.0
for g in gradients:
    v = beta * v + g
    print(f"gradient={g}, momentum v={v:.3f}")

# compare optimizers on the same toy problem
import torch
import torch.nn as nn

torch.manual_seed(0)
X = torch.randn(50, 2)
y = (X[:, 0] + X[:, 1] > 0).float().unsqueeze(1)

for name, opt_class, kwargs in [
    ("SGD", torch.optim.SGD, {"lr": 0.1}),
    ("SGD+momentum", torch.optim.SGD, {"lr": 0.1, "momentum": 0.9}),
    ("RMSprop", torch.optim.RMSprop, {"lr": 0.01}),
    ("Adam", torch.optim.Adam, {"lr": 0.01}),
    ("AdamW", torch.optim.AdamW, {"lr": 0.01, "weight_decay": 0.01}),
]:
    torch.manual_seed(0)
    model = nn.Sequential(nn.Linear(2, 1), nn.Sigmoid())
    optimizer = opt_class(model.parameters(), **kwargs)
    loss_fn = nn.BCELoss()
    for _ in range(50):
        optimizer.zero_grad()
        loss = loss_fn(model(X), y)
        loss.backward()
        optimizer.step()
    print(f"{name}: loss after 50 steps = {loss.item():.4f}")

#### Learning rate schedules

Why not just pick one fixed lr: a fixed learning rate is a compromise. Large enough to make fast progress early risks overshooting/oscillating once the loss gets close to its minimum; small enough for a precise final convergence is painfully slow early on, when big steps toward the minimum are still cheap and safe to take.

Step decay: multiply lr by a fixed factor every N epochs.
```
lr=0.1 for epochs 0-9, lr=0.1*0.5=0.05 for epochs 10-19, lr=0.025 for epochs 20-29, ...
```
Cosine annealing: smoothly decay lr following a cosine curve from an initial value down to (near) zero over the course of training, no abrupt jumps like step decay, the rate of decrease itself slows down near the end (cosine's shape flattens near 0), spending more time at small, precise steps right when it matters most.

Warmup: start with a SMALL lr and ramp UP for the first few steps, before switching to a decay schedule. Why: at initialization, weights are random (even with good init, see section 0.5), the very first gradients can be large and somewhat arbitrary, a big lr on these very first steps can push the model into a bad region it struggles to escape. Especially important for transformers (see `05-tokens-arch.ipynb`), which are known to be unstable to train without a warmup phase.

In [ ]:
import torch

model = torch.nn.Linear(2, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

step_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

print("step decay lr over epochs:")
opt2 = torch.optim.SGD(model.parameters(), lr=0.1)
sched2 = torch.optim.lr_scheduler.StepLR(opt2, step_size=10, gamma=0.5)
for epoch in [0, 9, 10, 19, 20, 29]:
    print(f"  epoch {epoch}: lr={opt2.param_groups[0]['lr']:.4f}")
    if epoch in [9, 19, 29]:
        sched2.step()

print("\ncosine annealing lr over epochs:")
opt3 = torch.optim.SGD(model.parameters(), lr=0.1)
sched3 = torch.optim.lr_scheduler.CosineAnnealingLR(opt3, T_max=30)
for epoch in range(30):
    if epoch in [0, 10, 20, 29]:
        print(f"  epoch {epoch}: lr={opt3.param_groups[0]['lr']:.4f}")
    sched3.step()

#### Dropout

Randomly zero out a fraction of neurons during training only, forces the network to not rely too heavily on any single neuron or co-adapted group of neurons, acts as an implicit ensemble of subnetworks, each training pass uses a different random subset of the network.

Worked example: a neuron with 100 inputs, each weight=1, each input=1, so neuron input = 100 with no dropout. At 40% dropout, roughly 40 of those 100 inputs get zeroed out during training, effective input drops to about 60. But at inference time, dropout is OFF, all 100 inputs contribute, giving 100, a train/inference mismatch in scale that needs correcting.

Fix, inverted dropout: sample a mask where each unit is kept with probability (1-p), zeroed with probability p, apply it, then rescale the kept units by 1/(1-p) to compensate.
```
p = 0.4 (40% dropout)
kept sum during training ~ 60
rescaled: 60 * (1/(1-0.4)) = 60 * 1.667 = 100
```
Training and inference now see the same expected scale, no correction needed at inference time, the model at eval time just uses every neuron with no zeroing and no rescaling. Common rates: 0.1-0.3 for large models, 0.4-0.5 for smaller fully-connected nets, too high a rate underfits, the network loses too much capacity per step to learn anything.

In [ ]:
import torch
import torch.nn as nn

dropout_layer = nn.Dropout(0.2)
p = 0.2
x = torch.randn((1, 5))

print("input:", x)
print("manual inverted-dropout scaling (x / (1-p)):", x / (1 - p))
print("nn.Dropout output (training mode):", dropout_layer(x))

dropout_layer.eval()
print("nn.Dropout output (eval mode, no drop, no scaling):", dropout_layer(x))

#### Batch normalization

Normalizes layer activations to mean 0/var 1 per mini-batch (with learned scale/shift), applied before activation. Stabilizes/accelerates training, allows higher LR, mild regularization effect.

#### Residual connections

y=F(x)+x. Gradient flows via the identity path (∂y/∂x=1+∂F/∂x) ⇒ mitigates vanishing gradients; easy to learn identity/small refinements ⇒ train 100+ layers; smoother loss landscape. Basis of ResNet and transformer blocks.

#### Exploding gradients and gradient clipping

Cause: repeated multiplication of weights >1 through depth/time (RNNs), bad init, high LR ⇒ NaN/loss spikes. Fixes: gradient clipping (norm), He/Xavier init, normalization layers, lower LR, residual connections, gated RNNs/transformers.

It caps the gradient norm so one rare huge gradient can't cause a destructive update — essential for RNNs and transformers. Caps gradient vector norm before the optimizer step — prevents an occasional huge gradient (common in RNNs, early training) from causing a destabilizing update.

#### Mini-batches and batch size

Batch: gradient over full dataset/update, stable but slow/memory-heavy. SGD: 1 sample/update, fast but noisy. Mini-batch (32-256 typical): standard tradeoff, GPU-parallel-friendly.

100 iterations per epoch → 500 iterations; batch size sets gradient noise, memory use and speed. Batch: one chunk of samples processed per weight update. Iteration: one weight update (one batch processed). Epoch: one full pass through the entire training dataset (many iterations).

Small batch: noisier gradient, acts as implicit regularization, tends toward flatter minima that generalize better. Large batch: smoother/lower-variance gradient, faster per-epoch wall time, tolerates (and needs) a larger LR — linear scaling rule — but without LR/regularization correction can converge to sharper, less-generalizing minima.

#### Early stopping and data augmentation

Early stopping: stop when validation loss stops improving (patience), restore best weights — limits effective capacity like L2. Augmentation: label-preserving transforms (flip/crop/noise, back-translation, mixup) enlarge effective data ⇒ less overfitting. Both cheap; augment train only.

### Distributed Training

How training scales across multiple devices once a single GPU can't hold the data, the model, or both: data parallelism (DDP), model parallelism, pipeline parallelism, and the sync-vs-async gradient aggregation choice. Distinct from `11-mlops.ipynb`'s PySpark section, which is about scaling ETL over a dataset, not distributing the training computation itself. See the training loop section above for the single-GPU loop this generalizes.

#### Data parallelism (DDP)

The model is small enough to fit on one device, but the DATASET or batch is large. Copy the full model onto every worker (GPU), split each batch into shards (one shard per worker), have each worker compute gradients on its own shard independently, then average ("all-reduce") the gradients across all workers before applying the update, so every worker ends the step with the exact same, updated model.

The averaged gradient from N equal-sized shards is mathematically identical to the gradient you'd have gotten from running the whole batch on one device, since averaging shard-level means of equal-sized shards is the same as the overall mean.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
x = rng.normal(size=8)
y = 2*x + rng.normal(scale=0.1, size=8)
w = 0.5  # current parameter

def mse_grad(xs, ys, w):
    preds = w*xs
    return np.mean(2*(preds - ys)*xs)

full_batch_grad = mse_grad(x, y, w)

# split into 2 shards (2 "workers"), compute each independently
g_worker1 = mse_grad(x[:4], y[:4], w)
g_worker2 = mse_grad(x[4:], y[4:], w)
allreduced_grad = (g_worker1 + g_worker2) / 2

print("full-batch gradient:       ", full_batch_grad)
print("worker 1 / worker 2 grads: ", g_worker1, g_worker2)
print("all-reduced (averaged):    ", allreduced_grad)
print("identical to full-batch:   ", np.isclose(full_batch_grad, allreduced_grad))

**What this doesn't fix.**

DDP scales how much DATA you can train on per step, not how big the MODEL itself can be, every worker still needs to hold a full copy of the model, optimizer state, and gradients in its own memory. Once the model itself is too large for one device's memory, you need model or pipeline parallelism instead.

#### Model parallelism vs. pipeline parallelism

Both split the MODEL (not the data) across devices, for when the model itself doesn't fit on one GPU.

- **Model (tensor) parallelism**: split an individual layer's computation across devices (e.g. each device holds a slice of a huge weight matrix, and they jointly compute one matrix multiply together). Devices must communicate on every single layer, so this needs very fast interconnects (e.g. NVLink within one server) and doesn't scale well across slower network links.
- **Pipeline parallelism**: assign different WHOLE LAYERS (or blocks of layers) to different devices, forming a pipeline, e.g. GPU 0 holds layers 1-8, GPU 1 holds layers 9-16, and so on. A batch is split into smaller micro-batches that flow through the pipeline stage by stage, like an assembly line.

Pipeline parallelism's cost is the "bubble": early in the pipeline, later stages sit idle waiting for the first micro-batch to arrive, and at the end, early stages sit idle waiting for the last micro-batch to finish. The bubble (idle) fraction for `P` pipeline stages and `M` micro-batches is `(P-1) / (M+P-1)`.

In [ ]:
def bubble_fraction(P, M):
    return (P - 1) / (M + P - 1)

for P, M in [(4, 4), (4, 8), (4, 16)]:
    print(f"P={P} stages, M={M} micro-batches -> idle bubble fraction = {round(bubble_fraction(P, M), 4)}")

**Why more micro-batches, not fewer.**

With only 4 micro-batches for 4 stages, 43% of every stage's time is wasted bubble (the pipeline hasn't finished filling/draining relative to how little useful work it does). Splitting the same batch into 16 micro-batches instead of 4 shrinks the bubble fraction to 16%, since the fixed fill/drain cost now amortizes over far more useful work. The tradeoff: more micro-batches means each one is smaller, which can hurt hardware utilization within a single micro-batch's own matrix multiplies, so this is tuned, not maximized blindly.

#### Sync vs. async gradient aggregation

**Synchronous** (what DDP's all-reduce above does): every worker's gradient is computed at the CURRENT, shared parameter value, and the update is applied only once every worker has finished and reported in. Simple, correct, but the whole step is only as fast as the SLOWEST worker (a straggler stalls everyone).

**Asynchronous**: workers push gradient updates whenever they finish, without waiting for others, and don't wait for a lock-step round either. No straggler problem, but a worker's gradient was computed against a parameter value that's now STALE, some other worker(s) may have already updated the shared parameters in the meantime. That staleness isn't just noise, it can change how a learning rate that's perfectly stable under synchronous updates behaves once you factor in that delay.

In [ ]:
lr = 0.6  # stable for synchronous gradient descent on this toy loss (see below)

# sync: gradient always computed at the CURRENT parameter value
w_sync = [10.0]
for t in range(10):
    grad = 2 * w_sync[-1]  # d/dw of loss = w^2
    w_sync.append(w_sync[-1] - lr * grad)
print("sync trajectory:            ", [round(v, 3) for v in w_sync])

# async: gradient computed from a value that's already 1 step STALE, applied to the current parameter
w_async = [10.0, 10.0]
for t in range(1, 11):
    stale_w = w_async[t - 1]
    grad = 2 * stale_w
    w_async.append(w_async[t] - lr * grad)
print("async (staleness=1) trajectory:", [round(v, 3) for v in w_async])

**Reading the result.**

At `lr=0.6`, synchronous updates converge cleanly to 0 within a handful of steps (oscillating but shrinking). The exact same learning rate under a 1-step-stale asynchronous update DIVERGES, the parameter oscillates with growing amplitude instead of shrinking. Nothing about the loss function or the learning rate changed, only the fact that the gradient was computed against an already-outdated parameter value. This is why async training in practice needs either a smaller learning rate than the synchronous-safe one, a hard cap on how stale a gradient is allowed to get before it's discarded, or simply isn't used at all for large-scale LLM pretraining, where synchronous data parallelism (DDP, ZeRO) is the standard despite the straggler cost.

### Transfer, Compression and Privacy

#### Transfer learning

Transfer learning: reuse a pretrained model's learned representations for a new (often frozen-backbone) task. Fine-tuning: continue training (some or all of) that pretrained model's weights on the new task's data — a specific way of doing transfer learning.

#### Model compression: pruning, quantization and distillation

Pruning: remove low-importance weights/channels — smaller, sparser model. Quantization: reduce numeric precision (FP32→INT8/4-bit) — smaller, faster inference, small quality cost. Distillation: train a smaller student to mimic a larger teacher's soft output distribution — new, purpose-built small model.

Soft label distribution carries more decision-boundary information (relative confidence across all classes) than a one-hot hard label — lets the student learn the teacher's "dark knowledge," not just its final answer.

#### Federated learning

Train a shared model across many decentralized devices/silos holding local data, without that raw data ever leaving its source — only model updates (gradients/weights) are aggregated centrally. Used when data can't be centralized for privacy/regulatory reasons (mobile devices, hospitals, banks) but a shared model still benefits from the combined signal.